# 01 - Exploratory Data Analysis
### Phishing URL Detector — PhiUSIIL Dataset

Goal: understand the dataset structure, class balance, and which features most strongly differentiate phishing URLs from legitimate ones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("darkgrid")
pd.set_option('display.max_columns', None)

df = pd.read_csv("../data/raw/PhiUSIIL_Phishing_URL_Dataset.csv")
print("Shape:", df.shape)
df.head()

## 1. Basic Info & Data Quality

In [ ]:
df.info()

In [ ]:
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate URLs:", df['URL'].duplicated().sum())

In [ ]:
# Drop duplicate URLs
before = df.shape[0]
df = df.drop_duplicates(subset='URL').reset_index(drop=True)
print(f"Dropped {before - df.shape[0]} duplicate rows. New shape: {df.shape}")

## 2. Label Distribution

**Note:** `label` = 1 -> legitimate, `label` = 0 -> phishing (verified by inspecting sample URLs).

In [ ]:
label_counts = df['label'].value_counts()
label_pct = df['label'].value_counts(normalize=True) * 100

print(label_counts)
print(label_pct)

fig, ax = plt.subplots(figsize=(5,4))
sns.countplot(data=df, x='label', palette=['#e74c3c', '#2ecc71'], ax=ax)
ax.set_xticklabels(['Phishing (0)', 'Legitimate (1)'])
ax.set_title("Class Distribution")
plt.show()

## 3. Key Feature Distributions: Phishing vs Legitimate

In [ ]:
key_features = [
    'URLLength', 'DomainLength', 'NoOfSubDomain', 'IsHTTPS',
    'HasTitle', 'HasFavicon', 'HasPasswordField', 'NoOfiFrame',
    'URLSimilarityIndex', 'TLDLegitimateProb'
]

fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for i, col in enumerate(key_features):
    sns.boxplot(data=df, x='label', y=col, ax=axes[i], palette=['#e74c3c', '#2ecc71'])
    axes[i].set_title(col)
    axes[i].set_xticklabels(['Phishing', 'Legitimate'])

plt.tight_layout()
plt.show()

## 4. Binary/Flag Features — Rate Comparison

In [ ]:
binary_features = [
    'IsDomainIP', 'IsHTTPS', 'HasObfuscation', 'HasTitle', 'HasFavicon',
    'Robots', 'IsResponsive', 'HasDescription', 'HasExternalFormSubmit',
    'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField',
    'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo'
]

rate_df = df.groupby('label')[binary_features].mean().T
rate_df.columns = ['Phishing (0)', 'Legitimate (1)']
rate_df = rate_df.sort_values('Legitimate (1)', ascending=False)

fig, ax = plt.subplots(figsize=(9, 8))
rate_df.plot(kind='barh', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_title("Proportion of URLs with Feature = 1, by Class")
ax.set_xlabel("Proportion")
plt.tight_layout()
plt.show()

rate_df

## 5. Correlation Heatmap (Numeric Features)

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, ax=ax)
ax.set_title("Correlation Heatmap — All Numeric Features")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each feature with the label (sorted)
corr_with_label = corr['label'].drop('label').sort_values(key=abs, ascending=False)
print("Top 15 features most correlated with label:")
corr_with_label.head(15)

## 6. Columns to Drop Before Modeling

Ye columns identifier/raw-text hain, direct model input ke liye useless ya leakage-prone:
- `FILENAME` — random file identifier
- `URL`, `Domain` — raw text (features already extracted separately)
- `TLD` — raw text (numeric proxies TLDLength, TLDLegitimateProb already exist)
- `Title` — raw scraped text, high cardinality

In [ ]:
cols_to_drop = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
print("Remaining feature count after drop:", df.shape[1] - len(cols_to_drop) - 1)  # -1 for label

# Save cleaned dataset for next notebook (02_preprocessing)
df.to_csv("../data/processed/eda_cleaned.csv", index=False)
print("Saved: ../data/processed/eda_cleaned.csv")

## Summary / Observations

Note your observations here after reviewing the EDA — e.g. which features looked like the strongest separators, any surprising patterns, and what to carry into the next notebook (`02_preprocessing.ipynb`).